# NLP-Based Resume–Job Alignment — Week 1

## Research question and hypothesis
Does combining semantic similarity with lexical matching improve identification of relevant resume–job qualifications compared with lexical matching alone?

My working hypothesis is that semantic information may help with differently worded qualifications. This notebook establishes synthetic inputs and initial lexical rankings only. Semantic matching and comparative ranking metrics remain planned.

## Execution instructions
Open this notebook from the repository root or its `src` folder and select a Python kernel. Install the packages listed in `src/requirements.txt` if needed. Restart the kernel, run all cells in order, and save the notebook. Outputs are written to `results/week_1/`.


In [1]:
from pathlib import Path
import json, platform
import numpy as np
import pandas as pd
import sklearn
def display(frame):
    print(frame.to_string(index=True))
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

ROOT = Path.cwd().resolve()
if ROOT.name == "src":
    ROOT = ROOT.parent
assert (ROOT / "src" / "week1_resume_job_foundation.ipynb").exists(), "Open from the extracted project root or src folder."
OUT = ROOT / "results" / "week_1"
OUT.mkdir(parents=True, exist_ok=True)
print("Project root:", ROOT)
print("Output folder:", OUT)
print("Python:", platform.python_version())

Project root: /home/user/tejaswi
Output folder: /home/user/tejaswi/results/week_1
Python: 3.12.8


## Synthetic inputs and relevance rubric
I construct fictional examples for data analysis, web development, and IT support. Each family includes direct wording, paraphrased wording, and partial qualifications. No personal resumes are used.

Grades are assigned before scores: 2 = substantial intended task/tool coverage, 1 = partial coverage, 0 = no relevant evidence in this toy scenario. These are author-defined scenario judgments, not independent annotations or hiring-suitability labels. Job-family grouping is deliberately simplified. Negative wording is retained to expose a limitation of word-based similarity.

In [2]:
jobs = pd.DataFrame([
("J1", "Data analyst", "Analyze data using Python and SQL. Create dashboards and communicate findings."),
("J2", "Web developer", "Build web interfaces using JavaScript HTML and CSS. Test responsive websites."),
("J3", "IT support", "Troubleshoot Windows computers and network connections. Resolve help desk tickets.")
], columns=["job_id", "title", "text"])
resumes = pd.DataFrame([
("R1", "Python and SQL data analysis. Created dashboards and presented findings."),
("R2", "Queried relational databases, analyzed datasets with pandas, and built visual reports for stakeholders."),
("R3", "Prepared spreadsheet reports and charts. No programming or database experience."),
("R4", "Built responsive websites with JavaScript HTML and CSS. Tested web interfaces."),
("R5", "Developed browser user interfaces with JS, styled pages with cascading style sheets, and checked layouts on phones."),
("R6", "Designed website mockups and page layouts. No coding experience."),
("R7", "Resolved help desk tickets, troubleshot Windows computers and network connections."),
("R8", "Diagnosed Microsoft desktop operating system faults, restored connectivity, and handled user service requests."),
("R9", "Assisted users with passwords and application installation. No network troubleshooting experience.")
], columns=["resume_id", "text"])
grades = {"J1":[2,2,1,0,0,0,0,0,0], "J2":[0,0,0,2,2,1,0,0,0], "J3":[0,0,0,0,0,0,2,2,1]}
labels = pd.DataFrame([(j,r,g) for j,gs in grades.items() for r,g in zip(resumes.resume_id,gs)], columns=["job_id","resume_id","relevance"])
display(jobs)
display(resumes)
display(labels.groupby(["job_id","relevance"]).size().unstack(fill_value=0))

  job_id          title                                                                                text
0     J1   Data analyst      Analyze data using Python and SQL. Create dashboards and communicate findings.
1     J2  Web developer       Build web interfaces using JavaScript HTML and CSS. Test responsive websites.
2     J3     IT support  Troubleshoot Windows computers and network connections. Resolve help desk tickets.
  resume_id                                                                                                                 text
0        R1                                             Python and SQL data analysis. Created dashboards and presented findings.
1        R2              Queried relational databases, analyzed datasets with pandas, and built visual reports for stakeholders.
2        R3                                      Prepared spreadsheet reports and charts. No programming or database experience.
3        R4                                       Bu

## Input validation
I check identifier integrity, text completeness, pair coverage, references, and grade values. Later checks test numerical behavior, not the validity of the scenario labels.

In [3]:
checks = {
"unique_job_ids": jobs.job_id.is_unique,
"unique_resume_ids": resumes.resume_id.is_unique,
"nonempty_text": bool(jobs.text.str.strip().ne("").all() and resumes.text.str.strip().ne("").all()),
"no_missing_values": not any(d.isna().any().any() for d in [jobs,resumes,labels]),
"unique_pairs": not labels.duplicated(["job_id","resume_id"]).any(),
"complete_pairs": len(labels) == len(jobs)*len(resumes),
"valid_references": set(labels.job_id)==set(jobs.job_id) and set(labels.resume_id)==set(resumes.resume_id),
"valid_grades": labels.relevance.isin([0,1,2]).all()
}
assert all(checks.values()), checks
print("Input checks passed:", len(checks))

Input checks passed: 8


## Initial lexical method
I fit unigram TF-IDF on the fixed resume collection only, transform the jobs using that vocabulary, and calculate cosine similarity. I retain the default tokenizer and no stop-word removal. Ties are resolved by resume ID. This is not a trained hiring model or a semantic-method comparison.

In [4]:
vectorizer = TfidfVectorizer(lowercase=True, ngram_range=(1,1))
resume_vectors = vectorizer.fit_transform(resumes.text)
job_vectors = vectorizer.transform(jobs.text)
scores = cosine_similarity(job_vectors, resume_vectors)
checks["finite_bounded_scores"] = bool(np.isfinite(scores).all() and (scores >= -1e-12).all() and (scores <= 1+1e-12).all())
checks["self_similarity"] = bool(np.isclose(cosine_similarity(resume_vectors[0],resume_vectors[0])[0,0],1))
checks["unknown_text_zero"] = bool(np.allclose(cosine_similarity(vectorizer.transform(["zzzxxyyqq"]),resume_vectors),0))
validation = pd.DataFrame([(k,bool(v)) for k,v in checks.items()],columns=["check","passed"])
assert validation.passed.all(), validation
rankings = pd.DataFrame([(j,r,float(scores[i,k])) for i,j in enumerate(jobs.job_id) for k,r in enumerate(resumes.resume_id)],columns=["job_id","resume_id","lexical_score"])
rankings = rankings.merge(labels,on=["job_id","resume_id"],validate="one_to_one").sort_values(["job_id","lexical_score","resume_id"],ascending=[True,False,True])
rankings["rank"] = rankings.groupby("job_id").cumcount()+1
display(validation)
display(rankings.loc[rankings["rank"]<=3])

                    check  passed
0          unique_job_ids    True
1       unique_resume_ids    True
2           nonempty_text    True
3       no_missing_values    True
4            unique_pairs    True
5          complete_pairs    True
6        valid_references    True
7            valid_grades    True
8   finite_bounded_scores    True
9         self_similarity    True
10      unknown_text_zero    True
   job_id resume_id  lexical_score  relevance  rank
0      J1        R1       0.806630          2     1
5      J1        R6       0.047172          0     2
2      J1        R3       0.044101          1     3
12     J2        R4       0.873329          2     1
13     J2        R5       0.084883          2     2
9      J2        R1       0.038268          0     3
24     J3        R7       0.879927          2     1
26     J3        R9       0.113592          1     2
18     J3        R1       0.038268          0     3


## Save verifiable outputs
I save inputs separately from computed rankings and validation results. Re-running overwrites these generated files. Environment versions describe the kernel used for this run, not a full environment lock.

In [5]:
summary = {"jobs":len(jobs),"resumes":len(resumes),"pairs":len(labels),"vocabulary_terms":len(vectorizer.vocabulary_),"checks_passed":int(validation.passed.sum()),"semantic_matching_executed":False,"comparative_evaluation_executed":False}
for name,frame in [("jobs",jobs),("resumes",resumes),("relevance_labels",labels),("lexical_rankings",rankings),("validation_checks",validation)]:
    frame.to_csv(OUT / (name+".csv"), index=False)
(OUT/"execution_summary.json").write_text(json.dumps(summary,indent=2)+"\n", encoding="utf-8")
(OUT/"environment.json").write_text(json.dumps({"python":platform.python_version(),"numpy":np.__version__,"pandas":pd.__version__,"scikit-learn":sklearn.__version__},indent=2)+"\n", encoding="utf-8")
print(json.dumps(summary,indent=2))
print("Saved files:", sorted(p.name for p in OUT.iterdir() if p.is_file()))

{
  "jobs": 3,
  "resumes": 9,
  "pairs": 27,
  "vocabulary_terms": 81,
  "checks_passed": 11,
  "semantic_matching_executed": false,
  "comparative_evaluation_executed": false
}
Saved files: ['environment.json', 'execution_summary.json', 'jobs.csv', 'lexical_rankings.csv', 'relevance_labels.csv', 'resumes.csv', 'validation_checks.csv']


## Preliminary interpretation and limitations
The direct-wording resume ranks first for each job in the initial run. For J1, irrelevant R6 ranks above partially relevant R3. These rankings illustrate a possible lexical limitation, not proof that semantic matching will improve results.

I have not executed semantic matching, selected hybrid weights, or computed comparative ranking metrics. The sample is tiny, labels are author-defined, generic words can distort rankings, and negative qualifications are not reliably interpreted. These synthetic results cannot establish real-world recruitment effectiveness.

## Week 2 plan
I will expand and freeze the dataset and relevance rubric, document a pretrained sentence-embedding model, predeclare the hybrid scoring rule, and implement Precision@K and NDCG with known-example tests. Any tuning will be separated from evaluation cases. I will inspect improvements and failures without changing labels to favor my hypothesis.

# Week 2 - Method implementation and baseline experiment

I am extending my Week 1 foundation to determine whether the proposed computational method can generate evidence relevant to my research question. My aim at this stage is to demonstrate a functioning comparison, not to complete the research or establish a final conclusion.

I will compare lexical matching, pretrained semantic similarity, and a fixed 50/50 hybrid using the same candidates. I examine both improvements and failures. I have not tuned the hybrid weight to these labels.

Run the Week 1 cells first, then execute the Week 2 cells below in order. Install `src/requirements_week2.txt` in the notebook kernel's environment and restart the kernel if necessary. The first model download requires internet; later runs use the cached files. The observations below summarize the recorded execution and the generated output files.


## 1. Extend and freeze my research inputs

I retain my original fictional examples and labels, then add nine fictional resumes: direct matches, partial matches, and keyword-heavy examples that describe nontechnical work. This gives 3 jobs, 18 resumes and 54 labeled pairs. These scenario labels represent my intended rubric, not independent hiring judgments. I set K = 3 and the equal hybrid weights before scoring.


In [6]:
"""Freeze a synthetic extension before computing Week 2 scores; no label tuning."""
import csv,json,hashlib
from pathlib import Path
ROOT=Path.cwd().resolve()
if ROOT.name == 'src': ROOT=ROOT.parent
assert (ROOT/'results/week_1/jobs.csv').exists(), 'Run Week 1 first from the repository root or src folder.'
D=ROOT/'data/week_2'; D.mkdir(parents=True,exist_ok=True)
def read(name):
 with (ROOT/'results/week_1'/name).open() as f: return list(csv.DictReader(f))
def write(name,rows,fields):
 with (D/name).open('w',newline='') as f:
  w=csv.DictWriter(f,fieldnames=fields);w.writeheader();w.writerows(rows)
jobs=read('jobs.csv');resumes=read('resumes.csv');labels=read('relevance_labels.csv')
extra=[
('R10','Used Python and SQL to clean sales datasets, build dashboards, and explain trends to managers.', 'J1',2,'direct'),
('R11','Prepared monthly business charts in spreadsheets; learning to query tables but no completed database projects.', 'J1',1,'partial'),
('R12','Sold software subscriptions to clients seeking Python SQL data analysis and dashboards; no hands-on analysis experience.',None,0,'keyword_negative'),
('R13','Implemented JavaScript components with HTML and CSS, tested mobile layouts and responsive web pages.','J2',2,'direct'),
('R14','Created page wireframes and visual prototypes; collaborated with developers without implementing browser code.','J2',1,'partial'),
('R15','Recruited JavaScript HTML CSS web developers; reviewed applicants but did not build or test websites.',None,0,'keyword_negative'),
('R16','Diagnosed Windows workstation failures, repaired network connections and resolved help desk support tickets.','J3',2,'direct'),
('R17','Helped colleagues reset account passwords and install desktop apps; escalated network faults to technicians.','J3',1,'partial'),
('R18','Marketed Windows computers and network equipment to help desk departments; did not troubleshoot technical issues.',None,0,'keyword_negative')]
metadata=[]
for rid,text,family,grade,kind in extra:
 resumes.append({'resume_id':rid,'text':text})
 metadata.append({'resume_id':rid,'scenario':kind,'intended_family':family or 'none','rationale':'Substantial task/tool coverage' if grade==2 else 'Partial task coverage' if grade==1 else 'Mentions target tools in nontechnical role; explicitly lacks task experience'})
 for job in jobs: labels.append({'job_id':job['job_id'],'resume_id':rid,'relevance':grade if job['job_id']==family else 0})
write('jobs.csv',jobs,['job_id','title','text']);write('resumes.csv',resumes,['resume_id','text']);write('relevance_labels.csv',labels,['job_id','resume_id','relevance']);write('added_scenarios.csv',metadata,['resume_id','scenario','intended_family','rationale'])
manifest={'rubric':{'0':'No demonstrated relevant task coverage in the constructed scenario','1':'Partial task coverage','2':'Substantial intended task and tool coverage'},'provenance':'Week 1 fictional inputs preserved; nine new fictional examples added before Week 2 scoring. All labels are author-defined, not independent judgments.','protocol':{'lexical_weight':0.5,'semantic_weight':0.5,'k':3,'relevant_threshold':1,'gain':'2**grade-1','tie_break':'resume_id ascending lexicographically','tuning':'none','model_id':'Xenova/all-MiniLM-L6-v2','model_revision':'751bff37182d3f1213fa05d7196b954e230abad9','model_file':'onnx/model.onnx','max_tokens':256},'sha256':{p.name:hashlib.sha256(p.read_bytes()).hexdigest() for p in sorted(D.glob('*.csv'))}}
(D/'manifest.json').write_text(json.dumps(manifest,indent=2)+'\n')
print('Frozen:',len(jobs),'jobs',len(resumes),'resumes',len(labels),'pairs')


Frozen: 3 jobs 18 resumes 54 pairs


### Input observations

I confirmed 3 jobs, 18 resumes and 54 unique labeled pairs. I retained the nine Week 1 resumes and added nine diagnostic examples covering direct qualifications, partial qualifications and keyword-heavy nontechnical roles. The labels remain scenario-based judgments. I fixed the input files and protocol before scoring, and the hash checks passed during execution.


## 2. Obtain the fixed pretrained model

I use the full-precision ONNX export of all-MiniLM-L6-v2, pinned to the revision recorded in my manifest. The download cell verifies both file hashes. This lets me run the pretrained sentence encoder on CPU without training a new model. I use attention-mask mean pooling and L2 normalization as described in the model documentation.

Model documentation: https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2

Pinned export: https://huggingface.co/Xenova/all-MiniLM-L6-v2/tree/751bff37182d3f1213fa05d7196b954e230abad9


In [7]:
"""Download the pinned full precision ONNX MiniLM export; never silently substitute."""
from pathlib import Path
import urllib.request
import json,hashlib
# ROOT was established in the input cell
config=json.loads((ROOT/'data/week_2/manifest.json').read_text())['protocol']
D=ROOT/'.cache/minilm';D.mkdir(parents=True,exist_ok=True)
expected={'model.onnx':'759c3cd2b7fe7e93933ad23c4c9181b7396442a2ed746ec7c1d46192c469c46e','tokenizer.json':'da0e79933b9ed51798a3ae27893d3c5fa4a201126cef75586296df9b4d2c62a0'}
for source,dest in [('tokenizer.json','tokenizer.json'),(config['model_file'],'model.onnx')]:
 p=D/dest
 if not p.exists():
  url=f"https://huggingface.co/{config['model_id']}/resolve/{config['model_revision']}/{source}"
  print('Downloading',source,flush=True)
  tmp=p.with_suffix('.part');urllib.request.urlretrieve(url,tmp);tmp.replace(p)
 digest=hashlib.sha256(p.read_bytes()).hexdigest()
 if digest!=expected[dest]: raise ValueError('Model asset digest mismatch: '+dest)
 print(dest,p.stat().st_size,digest,flush=True)


tokenizer.json 711661 da0e79933b9ed51798a3ae27893d3c5fa4a201126cef75586296df9b4d2c62a0
model.onnx 90387606 759c3cd2b7fe7e93933ad23c4c9181b7396442a2ed746ec7c1d46192c469c46e


## 3. Implement my scoring and evaluation methods

I keep the Week 1 unigram TF-IDF method as my lexical baseline. Semantic similarity is cosine similarity between normalized sentence embeddings. My proposed hybrid is `0.5 * lexical_score + 0.5 * semantic_score`. I also retain semantic-only scoring as a reference condition.

Precision@3 treats grades 1 and 2 as relevant. NDCG@3 uses gain `2**grade - 1` and discounts by `log2(rank + 1)`. I break score ties by resume ID. Labels are used only for evaluation, not to compute scores.


In [8]:
"""Scoring and ranking metrics. Relevance labels never enter scoring functions."""
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

def lexical_scores(jobs,resumes):
 v=TfidfVectorizer(lowercase=True,ngram_range=(1,1))
 r=v.fit_transform(resumes)
 return cosine_similarity(v.transform(jobs),r)

def masked_mean(hidden,mask):
 m=np.asarray(mask,dtype=np.float32)[...,None]
 pooled=(hidden*m).sum(axis=1)/np.maximum(m.sum(axis=1),1e-9)
 return pooled/np.maximum(np.linalg.norm(pooled,axis=1,keepdims=True),1e-12)

def embeddings(texts,model_dir):
 from tokenizers import Tokenizer
 import onnxruntime as ort
 tokenizer=Tokenizer.from_file(str(model_dir/'tokenizer.json'))
 tokenizer.enable_truncation(max_length=256)
 tokenizer.enable_padding(pad_id=0,pad_token='[PAD]')
 encoded=tokenizer.encode_batch(texts)
 inputs={'input_ids':np.array([e.ids for e in encoded],dtype=np.int64),'attention_mask':np.array([e.attention_mask for e in encoded],dtype=np.int64),'token_type_ids':np.array([e.type_ids for e in encoded],dtype=np.int64)}
 opts=ort.SessionOptions(); opts.intra_op_num_threads=1;opts.inter_op_num_threads=1
 session=ort.InferenceSession(str(model_dir/'model.onnx'),sess_options=opts,providers=['CPUExecutionProvider'])
 hidden=session.run(None,{i.name:inputs[i.name] for i in session.get_inputs()})[0]
 return masked_mean(hidden,inputs['attention_mask']),sum(len(e.ids)>=256 for e in encoded)

def hybrid_scores(lexical,semantic,weight=.5):
 if not 0<=weight<=1: raise ValueError('weight must lie in [0,1]')
 return weight*lexical+(1-weight)*semantic

def ranking(scores,ids):
 return sorted(range(len(ids)),key=lambda i:(-float(scores[i]),ids[i]))

def precision_at_k(grades,k=3):
 if k<1 or len(grades)<k: raise ValueError('k must be positive and <= candidate count')
 return float(np.count_nonzero(np.asarray(grades[:k])>=1)/k)

def ndcg_at_k(grades,k=3):
 if k<1 or len(grades)<k: raise ValueError('k must be positive and <= candidate count')
 def dcg(g): return float(np.sum((2.**np.asarray(g[:k])-1)/np.log2(np.arange(2,k+2))))
 ideal=dcg(sorted(grades,reverse=True))
 return dcg(grades)/ideal if ideal else 0.


## 4. Validate important calculations

I test known examples before interpreting the comparison. I check Precision@3, perfect and zero NDCG, a hand-calculated NDCG example, agreement with scikit-learn, tie-breaking, masked pooling, hybrid endpoint identities, unknown vocabulary and invalid K. Passing these checks does not validate my relevance judgments.


In [9]:
import io
import unittest
import numpy as np
from sklearn.metrics import ndcg_score
class MethodTests(unittest.TestCase):
 def test_precision_known(self): self.assertAlmostEqual(precision_at_k([2,0,1]),2/3)
 def test_ndcg_perfect(self): self.assertEqual(ndcg_at_k([2,2,1,0]),1)
 def test_ndcg_zero(self): self.assertEqual(ndcg_at_k([0,0,0]),0)
 def test_ndcg_known(self): self.assertAlmostEqual(ndcg_at_k([0,2,1]),(3/np.log2(3)+.5)/(3+1/np.log2(3)))
 def test_reference_ndcg(self):
  grades=[0,2,1,2,0];score=[5,4,3,2,1]
  self.assertAlmostEqual(ndcg_at_k(grades),ndcg_score([2**np.array(grades)-1],[score],k=3))
 def test_ties(self): self.assertEqual(ranking([.2,.2],['R2','R1']),[1,0])
 def test_pool_padding(self): np.testing.assert_allclose(masked_mean(np.array([[[3.,4.],[99.,99.]]]),[[1,0]]),[[.6,.8]])
 def test_hybrid_endpoints(self):
  np.testing.assert_allclose(hybrid_scores(np.array([.2]),np.array([.8]),1),[.2])
  np.testing.assert_allclose(hybrid_scores(np.array([.2]),np.array([.8]),0),[.8])
 def test_unknown_vocabulary(self): np.testing.assert_allclose(lexical_scores(['zzzxxyyqq'],['python analysis','network repair']),0)
 def test_invalid_k(self):
  with self.assertRaises(ValueError): precision_at_k([1],3)
  with self.assertRaises(ValueError): ndcg_at_k([1],0)
suite = unittest.defaultTestLoader.loadTestsFromTestCase(MethodTests)
validation_stream = io.StringIO()
validation_result = unittest.TextTestRunner(stream=validation_stream, verbosity=2).run(suite)
print(validation_stream.getvalue())
assert validation_result.wasSuccessful(), "Resolve validation failures before interpreting results"


test_hybrid_endpoints (builtins.MethodTests.test_hybrid_endpoints) ... ok
test_invalid_k (builtins.MethodTests.test_invalid_k) ... ok
test_ndcg_known (builtins.MethodTests.test_ndcg_known) ... ok
test_ndcg_perfect (builtins.MethodTests.test_ndcg_perfect) ... ok
test_ndcg_zero (builtins.MethodTests.test_ndcg_zero) ... ok
test_pool_padding (builtins.MethodTests.test_pool_padding) ... ok
test_precision_known (builtins.MethodTests.test_precision_known) ... ok
test_reference_ndcg (builtins.MethodTests.test_reference_ndcg) ... ok
test_ties (builtins.MethodTests.test_ties) ... ok
test_unknown_vocabulary (builtins.MethodTests.test_unknown_vocabulary) ... ok

----------------------------------------------------------------------
Ran 10 tests in 0.005s

OK



### Validation observations

All 10 method tests passed. These include known-value ranking metrics, agreement with scikit-learn NDCG, attention-mask pooling, tie-breaking and hybrid endpoint behavior. The experiment's input-integrity and numerical checks also passed. No input reached the 256-token limit. These checks support correct execution of the implemented calculations; they do not establish that my relevance labels are independently valid.


## 5. Execute the initial controlled comparison

I rank the same 18 candidates for each of the three jobs using all three methods. I do not perform weight selection, model training, or fine-tuning. This is a small synthetic development experiment; it is not an independent held-out evaluation.

The cell checks input hashes and pair integrity, verifies numerical behavior, and saves complete rankings, per-job metrics, macro averages, paired differences and execution provenance under `results/week_2/`. This is an initial experiment to demonstrate that the method produces measurable evidence.


In [10]:
"""Run the frozen Week 2 protocol from any working directory."""
from pathlib import Path
import json,hashlib,platform,importlib.metadata,time
import numpy as np
import pandas as pd
D=ROOT/'data/week_2';O=ROOT/'results/week_2';O.mkdir(parents=True,exist_ok=True)
manifest=json.loads((D/'manifest.json').read_text());p=manifest['protocol']
for name,h in manifest['sha256'].items():
 if hashlib.sha256((D/name).read_bytes()).hexdigest()!=h: raise ValueError('Frozen input changed: '+name)
jobs=pd.read_csv(D/'jobs.csv');resumes=pd.read_csv(D/'resumes.csv');labels=pd.read_csv(D/'relevance_labels.csv')
assert jobs.job_id.is_unique and resumes.resume_id.is_unique
assert all(not f.isna().any().any() for f in [jobs,resumes,labels])
assert jobs.text.str.strip().ne('').all() and resumes.text.str.strip().ne('').all()
assert not labels.duplicated(['job_id','resume_id']).any()
assert set(zip(labels.job_id,labels.resume_id))=={(j,r) for j in jobs.job_id for r in resumes.resume_id}
assert labels.relevance.isin([0,1,2]).all()
started=time.perf_counter();lex=lexical_scores(jobs.text.tolist(),resumes.text.tolist());lex_time=time.perf_counter()-started
started=time.perf_counter();vectors,truncated=embeddings(jobs.text.tolist()+resumes.text.tolist(),ROOT/'.cache/minilm')
sem=vectors[:len(jobs)]@vectors[len(jobs):].T;sem_time=time.perf_counter()-started
assert vectors.shape==(len(jobs)+len(resumes),384)
assert np.isfinite(vectors).all() and np.allclose(np.linalg.norm(vectors,axis=1),1,atol=1e-5)
scores={'lexical':lex,'semantic':sem,'hybrid':hybrid_scores(lex,sem,p['lexical_weight'])}
rows=[];metrics=[]
grade_lookup=labels.set_index(['job_id','resume_id']).relevance
for method,values in scores.items():
 assert np.isfinite(values).all() and (values>=-1.00001).all() and (values<=1.00001).all()
 for i,job in enumerate(jobs.job_id):
  order=ranking(values[i],resumes.resume_id.tolist());grades=[]
  for rank,index in enumerate(order,1):
   rid=resumes.resume_id.iloc[index];g=int(grade_lookup.loc[(job,rid)]);grades.append(g)
   rows.append({'method':method,'job_id':job,'resume_id':rid,'rank':rank,'score':float(values[i,index]),'relevance':g})
  metrics.append({'method':method,'job_id':job,'precision_at_3':precision_at_k(grades,p['k']),'ndcg_at_3':ndcg_at_k(grades,p['k'])})
rankings=pd.DataFrame(rows);m=pd.DataFrame(metrics);agg=m.groupby('method')[['precision_at_3','ndcg_at_3']].mean().reset_index()
rankings.to_csv(O/'rankings.csv',index=False);m.to_csv(O/'metrics_by_job.csv',index=False);agg.to_csv(O/'metrics_summary.csv',index=False)
comparison=m.pivot(index='job_id',columns='method',values=['precision_at_3','ndcg_at_3']);comparison.columns=['_'.join(c) for c in comparison.columns]
for metric in ['precision_at_3','ndcg_at_3']:comparison[metric+'_hybrid_minus_lexical']=comparison[metric+'_hybrid']-comparison[metric+'_lexical']
comparison.to_csv(O/'paired_comparison.csv')
# Direct-wording and keyword-negative cases expose both improvements and failures.
rankings[rankings['rank']<=3].to_csv(O/'top3_cases.csv',index=False)
summary={'jobs':len(jobs),'resumes':len(resumes),'pairs':len(labels),'methods':list(scores),'model_id':p['model_id'],'model_revision':p['model_revision'],'input_sha256':manifest['sha256'],'model_sha256':{n:hashlib.sha256((ROOT/'.cache/minilm'/n).read_bytes()).hexdigest() for n in ['model.onnx','tokenizer.json']},'truncated_texts':truncated,'lexical_seconds':lex_time,'semantic_seconds_including_session_load':sem_time,'protocol':p,'seed':'not applicable: no sampling, training or tuning; deterministic CPU inference and fixed inputs','environment':{'python':platform.python_version(),**{n:importlib.metadata.version(n) for n in ['numpy','pandas','scikit-learn','onnxruntime','tokenizers']}}}
(O/'execution_summary.json').write_text(json.dumps(summary,indent=2)+'\n')
print(agg.to_string(index=False));print('\nPaired differences:\n',comparison.to_string())
(O/"validation_tests.txt").write_text(validation_stream.getvalue())


  method  precision_at_3  ndcg_at_3
  hybrid        0.666667   0.744880
 lexical        0.666667   0.724399
semantic        0.777778   0.823093

Paired differences:
         precision_at_3_hybrid  precision_at_3_lexical  precision_at_3_semantic  ndcg_at_3_hybrid  ndcg_at_3_lexical  ndcg_at_3_semantic  precision_at_3_hybrid_minus_lexical  ndcg_at_3_hybrid_minus_lexical
job_id                                                                                                                                                                                                      
J1                   0.666667                0.666667                 0.666667          0.703918           0.703918            0.703918                                  0.0                        0.000000
J2                   0.666667                0.666667                 1.000000          0.765361           0.703918            1.000000                                  0.0                        0.061443
J3            

## 6. Inspect the candidate-level results

I inspect the results alongside the averages. In particular, I check whether paraphrased qualifications are retrieved and whether the methods still reward technical keywords in resumes that explicitly lack the relevant task experience.


In [11]:
print("Top three candidates for each method and job:")
print(rankings.loc[rankings["rank"] <= 3].to_string(index=False))
print("\nOriginal paraphrased examples R2, R5 and R8:")
print(rankings.loc[rankings.resume_id.isin(["R2", "R5", "R8"])].to_string(index=False))
print("\nKeyword-negative examples R12, R15 and R18:")
print(rankings.loc[rankings.resume_id.isin(["R12", "R15", "R18"])].to_string(index=False))


Top three candidates for each method and job:
  method job_id resume_id  rank    score  relevance
 lexical     J1        R1     1 0.785905          2
 lexical     J1       R12     2 0.378022          0
 lexical     J1       R10     3 0.337294          2
 lexical     J2        R4     1 0.726902          2
 lexical     J2       R15     2 0.564144          0
 lexical     J2       R13     3 0.443216          2
 lexical     J3        R7     1 0.774571          2
 lexical     J3       R16     2 0.536201          2
 lexical     J3       R18     3 0.523665          0
semantic     J1        R1     1 0.877323          2
semantic     J1       R12     2 0.730118          0
semantic     J1       R10     3 0.714541          2
semantic     J2        R4     1 0.906794          2
semantic     J2       R13     2 0.704390          2
semantic     J2        R5     3 0.637016          2
semantic     J3        R7     1 0.841569          2
semantic     J3       R16     2 0.805644          2
semantic     J3   

## My preliminary observations

### Execution and baseline comparison

The Week 2 cells completed successfully using the pinned MiniLM model. I evaluated all 54 pairs with lexical, semantic and hybrid scoring. The saved outputs record the execution environment and input/model hashes.

| Method | Macro Precision@3 | Macro NDCG@3 |
|---|---:|---:|
| Lexical baseline | 0.666667 | 0.724399 |
| Semantic only | 0.777778 | 0.823093 |
| Equal-weight hybrid | 0.666667 | 0.744880 |

The hybrid increased macro NDCG@3 by 0.020481 compared with the lexical baseline. Its Precision@3 remained unchanged. Semantic-only scoring had the highest averages in this initial experiment.

### Per-job differences

| Job | Lexical P@3 | Hybrid P@3 | Semantic P@3 | Lexical NDCG@3 | Hybrid NDCG@3 | Semantic NDCG@3 |
|---|---:|---:|---:|---:|---:|---:|
| J1 - Data analyst | 0.666667 | 0.666667 | 0.666667 | 0.703918 | 0.703918 | 0.703918 |
| J2 - Web developer | 0.666667 | 0.666667 | 1.000000 | 0.703918 | 0.765361 | 1.000000 |
| J3 - IT support | 0.666667 | 0.666667 | 0.666667 | 0.765361 | 0.765361 | 0.765361 |

The hybrid's top-three metric improvement came entirely from J2. It moved the grade-0 recruiting example R15 from rank 2 to rank 3 while moving grade-2 R13 from rank 3 to rank 2. J1 and J3 had no top-three metric changes. An average improvement therefore does not indicate improvement across every job.

### Paraphrased qualifications

For J2, the paraphrased resume R5 moved from lexical rank 5 to semantic rank 3 and hybrid rank 4. The semantic method brought it into the first three; the hybrid did not. For J1, R2 moved from rank 12 under lexical scoring to rank 4 under both semantic and hybrid scoring. These examples show that the implemented semantic component can change rankings for differently worded qualifications, although not every change improves a top-three metric.

### Remaining failure cases

R12 describes software sales and explicitly lacks hands-on data-analysis experience, yet it remained rank 2 for J1 under all three methods. Similarly, the IT-equipment sales example R18 remained rank 3 for J3. Technical vocabulary and topical similarity can still outweigh the absence of relevant task experience. I have not corrected these cases by changing the labels or selecting a new weight after inspecting the results.

### Relation to my research question

This experiment demonstrates progress beyond my Week 1 lexical foundation: the semantic and hybrid methods execute, the baseline comparison produces measurable evidence, and the outputs expose both improvements and failures. The results provide mixed preliminary support for the hypothesis. The hybrid slightly improved ordering on one job but did not increase Precision@3 and did not outperform semantic-only scoring. I cannot conclude that combining the methods is generally superior.

## Limitations and next work

My inputs are synthetic, there are only three job queries, and the relevance labels are not independently validated. The measurements above are actual computational outputs on those inputs, not results from real applicants or recruitment activity. Near-duplicate direct matches and constructed job families further limit generalization. I performed no model training, weight tuning or independent held-out evaluation, and these three queries do not support a meaningful statistical superiority claim.

For the next milestone, I will focus on a manageable extension: examine more varied cases, strengthen the relevance rubric and independent judgments where feasible, and separate development queries from evaluation queries before any weight tuning. The present milestone establishes an initial evidence-producing process; it does not complete the research.
